**This code is to use shape descriptors for classification using multi-layer perceptrons (MLPs):**

In [1]:
import os
import ast
import json
import random
import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA, FastICA, NMF, TruncatedSVD
from sklearn.random_projection import GaussianRandomProjection, SparseRandomProjection
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [2]:
# ---------------- CONFIG ----------------
MODEL = "padc_fisvista_dim_search"
DEVICE_ID = 0
SEED = 42
BATCH_SIZE = 512
EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-5
DROPOUT = 0.3
EXPLAINED_VARIANCE_THRESHOLD = 0.99

DATA_DIR = r'/home/c/choton/beemachine/datasets/Others/fish-vista'
LOG_DIR = f"./{MODEL}_logs/"
train_path = r"/home/c/choton/beemachine/codes/AG_vision_2026/4_shape_feature_analysis/FishVista/shape_features_fishvista/fishvista_shape_features_train.csv"
val_path = r"/home/c/choton/beemachine/codes/AG_vision_2026/4_shape_feature_analysis/FishVista/shape_features_fishvista/fishvista_shape_features_val.csv"
test_path = r"/home/c/choton/beemachine/codes/AG_vision_2026/4_shape_feature_analysis/FishVista/shape_features_fishvista/fishvista_shape_features_test.csv"

device = torch.device(f"cuda:{DEVICE_ID}" if torch.cuda.is_available() else "cpu")

# Read the mask labels (traits)
seg_json_path = os.path.join(DATA_DIR, 'segmentation_masks', 'seg_id_trait_map.json')
with open(seg_json_path, 'r') as json_file:
    content = json_file.read()
    seg_json = ast.literal_eval(content)
    print('Names of the mask labels (traits):')
    print(seg_json)
labels = list(seg_json.values())

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

Names of the mask labels (traits):
{0: 'Background', 1: 'Head', 2: 'Eye', 3: 'Dorsal fin', 4: 'Pectoral fin', 5: 'Pelvic fin', 6: 'Anal fin', 7: 'Caudal fin', 8: 'Adipose fin', 9: 'Barbel'}


In [3]:
# ---------------- FAST IMPUTATION ----------------
def fill_by_class_mean(df, class_col="label"):
    df = df.copy()

    # Replace zeros with NaN
    df.replace(0, np.nan, inplace=True)

    # Keep only columns that are not all NaN
    df = df.dropna(axis=1, how="all")

    numeric_cols = df.select_dtypes(include=[np.number]).columns

    # Compute class means once
    class_means = df.groupby(class_col)[numeric_cols].transform("mean")

    # Fill NaNs with class means
    df[numeric_cols] = df[numeric_cols].fillna(class_means)

    # Global fallback
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

    return df

# ---------------- DATASET ----------------
class ShapeFeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# The MLP classifier Network
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.3):
        super().__init__()
        mid_dim = input_dim
        if (input_dim // 2) > num_classes:
            mid_dim = input_dim // 2
        self.net = nn.Sequential(
            nn.Linear(input_dim, mid_dim),
            nn.LayerNorm(mid_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mid_dim, num_classes)
        )
        # simple init
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

def reduce_dimensions(X_train, X_val, X_test, method="PCA", n_components=100, random_state=SEED):
    """
    Apply dimensionality reduction on the dataset.
    Supported methods: "PCA", "ICA", "NMF", "TruncatedSVD", "GRP", "SRP"
    """
    if method == "PCA":
        reducer = PCA(n_components=n_components, random_state=random_state)
    elif method == "ICA":
        reducer = FastICA(n_components=n_components, random_state=random_state, max_iter=1000)
    elif method == "NMF":
        # NMF requires non-negative input
        X_train_nonneg = np.clip(X_train, 0, None)
        X_val_nonneg = np.clip(X_val, 0, None)
        X_test_nonneg = np.clip(X_test, 0, None)
        reducer = NMF(n_components=n_components, init='nndsvda', random_state=random_state, max_iter=1000)
        X_train = reducer.fit_transform(X_train_nonneg)
        X_val = reducer.transform(X_val_nonneg)
        X_test = reducer.transform(X_test_nonneg)
        return X_train, X_val, X_test, reducer
    elif method == "TruncatedSVD":
        reducer = TruncatedSVD(n_components=n_components, random_state=random_state)
    elif method == "GRP":
        reducer = GaussianRandomProjection(n_components=n_components, random_state=random_state)
    elif method == "SRP":
        reducer = SparseRandomProjection(n_components=n_components, random_state=random_state)
    else:
        raise ValueError(f"Unknown dimensionality reduction method: {method}")
    
    X_train_reduced = reducer.fit_transform(X_train)
    X_val_reduced   = reducer.transform(X_val)
    X_test_reduced  = reducer.transform(X_test)
    
    return X_train_reduced, X_val_reduced, X_test_reduced, reducer

def select_n_components_via_pca(X_train, explained_var_threshold=0.95, random_state=SEED, plot=False):
    """Use PCA to select optimal n_components based on explained variance."""
    pca_full = PCA(n_components=X_train.shape[1], random_state=random_state)
    pca_full.fit(X_train)
    cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
    n_components_opt = np.argmax(cumulative_variance >= explained_var_threshold) + 1
    print(f"Optimal n_components (PCA, {explained_var_threshold*100}% variance): {n_components_opt}")
    
    # Optional plot
    if plot:
        plt.figure(figsize=(8,5))
        plt.plot(cumulative_variance, marker='o')
        plt.axhline(y=explained_var_threshold, color='r', linestyle='--')
        plt.xlabel('Number of components')
        plt.ylabel('Cumulative explained variance')
        plt.title('PCA Explained Variance')
        plt.grid(True)
        plt.show()    
    return n_components_opt


# Training functions
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)

            total_loss += loss.item() * X.size(0)

            preds = outputs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / total, correct / total

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, epochs, log_dir):
    # Full training loop with logging and checkpointing 
    history = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    checkpoints_dir = os.path.join(log_dir, "checkpoints")
    os.makedirs(checkpoints_dir, exist_ok=True)

    # Initialize validation loss and start training
    best_val_loss = float("inf")
    print("Starting the first epoch...")

    for epoch in range(epochs):
        # ---- Training ----
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

        # ---- Validation ----
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        # ---- Scheduler ----
        scheduler.step(val_loss)

        # ---- Logging ----
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        # ---- Checkpoint ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(checkpoints_dir, "best_model.pth")
            torch.save(model.state_dict(), best_path)

            checkpoint_path = os.path.join(checkpoints_dir, f"checkpoint_epoch{epoch+1}.pth")
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss
            }, checkpoint_path)
        
        print(f"\r Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")    
    print("Training complete!")

    # ---- Save logs ----
    df = pd.DataFrame(history)
    csv_logfile = os.path.join(log_dir, "training_log.csv")
    json_logfile = os.path.join(log_dir, "training_log.json")

    df.to_csv(csv_logfile, index=False)
    with open(json_logfile, "w") as f:
        json.dump(history, f, indent=4)

    return history

In [4]:
train_df = pd.read_csv(train_path) 
val_df = pd.read_csv(val_path) 
test_df = pd.read_csv(test_path)
train_df

,image,class_id,part1_area,part1_perimeter,part1_aspect_ratio,part1_extent,part1_solidity,part1_eccentricity,part1_orientation,part1_circularity,...,area_ratio_part8_to_part9,area_ratio_part1_to_total,area_ratio_part2_to_total,area_ratio_part3_to_total,area_ratio_part4_to_total,area_ratio_part5_to_total,area_ratio_part6_to_total,area_ratio_part7_to_total,area_ratio_part8_to_total,area_ratio_part9_to_total
0,INHS_FISH_107832.jpg,1173,8253.0,479.504608,2.180382,0.665726,0.895994,0.888625,-0.040064,0.451062,...,0.258427,0.322081,0.003434,0.137254,0.078130,0.088042,0.105526,0.248049,0.003590,0.013893
1,INHS_FISH_25132.jpg,1102,4780.0,539.847778,2.999221,0.567224,0.785409,0.942778,0.053063,0.206108,...,NaN,0.265526,0.062382,0.062715,0.073047,0.058938,0.069826,0.407566,NaN,NaN
2,INHS_FISH_40481.jpg,1153,4291.0,467.569580,2.510842,0.563567,0.807794,0.917267,-0.095715,0.246647,...,NaN,0.242635,0.046424,0.124173,0.062991,0.086797,0.074696,0.362284,NaN,NaN
3,ark-_65665_m38f1c86b3282243d9b4b96d24f1983f3d.jpg,1291,10604.0,625.487366,2.746281,0.596568,0.908655,0.931349,-0.074494,0.340598,...,NaN,0.478844,0.020682,0.030752,0.111357,0.013547,0.036216,0.308602,NaN,NaN
4,INHS_FISH_15401.jpg,611,7436.0,578.315796,2.288819,0.563333,0.844424,0.899507,-0.091407,0.279395,...,NaN,0.336364,0.047451,0.046094,0.107387,0.012349,0.079251,0.371104,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39795,1484297470887I63P0bqYUYhsBtCS#0.jpg,1019,7411.0,508.255890,3.164971,0.600081,0.892139,0.948773,-0.177956,0.360514,...,NaN,0.404729,0.004915,0.252089,0.120474,0.028398,0.029654,0.159740,NaN,NaN
39796,BMNH 1852.2.19.126-127 Barbus longispinis e-S.jpg,1708,5333.0,437.084320,1.984089,0.646738,0.877282,0.863698,0.035413,0.350793,...,NaN,0.298450,0.031115,0.174324,0.058369,0.045610,0.073591,0.317197,NaN,0.001343
39797,BMNH_1923-4-30-251_02-S.jpg,833,6081.0,487.717834,2.433160,0.599704,0.865746,0.911641,-0.069655,0.321253,...,NaN,0.344396,0.041740,0.149402,0.098488,0.009458,0.091069,0.265447,NaN,NaN
39798,UWZM-F-0003627.JPG,1186,7147.0,610.226440,3.272322,0.635176,0.858705,0.952162,0.024923,0.241186,...,NaN,0.445935,0.062519,0.046172,0.079366,0.019155,0.000125,0.338242,0.008486,NaN


In [5]:
print(f"Shape of dataset, train: {train_df.shape}, val: {val_df.shape}, test: {test_df.shape}")

Shape of dataset, train: (39800, 2377), val: (6779, 2377), test: (9781, 2377)


**Total number of descriptors for Fish-Vista dataset with 10 parts** (including background): $total = 233*n + \binom{n-1}{2} + (n-1) = 233*10 + \binom{9}{2} + 9 = 2330+36+9 = 2375$. Total number of columns will be 2 more (check the shape in previous cell).

In [6]:
train_df = fill_by_class_mean(train_df, class_col="class_id")
val_df   = fill_by_class_mean(val_df, class_col="class_id")
test_df  = fill_by_class_mean(test_df, class_col="class_id")

In [7]:
feature_cols = train_df.drop(columns=["image", "class_id"]).columns

X_train = train_df[feature_cols].values
X_val   = val_df[feature_cols].values
X_test  = test_df[feature_cols].values

y_train = train_df["class_id"].values
y_val   = val_df["class_id"].values
y_test  = test_df["class_id"].values

# Standardize
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_val_std   = scaler.transform(X_val)
X_test_std  = scaler.transform(X_test)

classes = np.unique(y_train)

# --- Sanity check ---
assert not np.isnan(X_train_std).any(), "NaNs remain in X_train_std!"
assert not np.isnan(X_val_std).any(), "NaNs remain in X_val_std!"
assert not np.isnan(X_test_std).any(), "NaNs remain in X_test_std!"
print("✅ Class-wise NaN imputation complete — no missing values remain.")

✅ Class-wise NaN imputation complete — no missing values remain.


In [8]:
# Fit PCA with all components
pca_full = PCA(n_components=X_train_std.shape[1], random_state=SEED)
pca_full.fit(X_train_std)

# Cumulative explained variance
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

# Find number of components to explain desired variance (e.g., 95%)
explained_threshold = EXPLAINED_VARIANCE_THRESHOLD
n_components_opt = np.argmax(cumulative_variance >= explained_threshold) + 1

print(f"Optimal n_components for {explained_threshold*100}% variance: {n_components_opt}")

Optimal n_components for 99.0% variance: 1640


In [9]:
methods = ["PCA", "ICA", "TruncatedSVD", "GRP", "SRP", "NMF"]
best_acc = -1
best_method = None
best_data = None

device = torch.device(f"cuda:{DEVICE_ID}")

summary_rows = []

for method in methods:
    print(f"\n================ {method} ================\n")

    Xtr, Xva, Xte, reducer = reduce_dimensions(
        X_train_std, X_val_std, X_test_std,
        method=method,
        n_components=n_components_opt
    )

    train_dataset = ShapeFeatureDataset(Xtr, y_train)
    val_dataset   = ShapeFeatureDataset(Xva, y_val)
    test_dataset  = ShapeFeatureDataset(Xte, y_test)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = MLPClassifier(Xtr.shape[1], len(classes), dropout=DROPOUT).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    history = train_model(
        model, train_loader, val_loader,
        criterion, optimizer, scheduler,
        device, EPOCHS,
        LOG_DIR + f"_{method}"
    )

    # ---- load best checkpoint ----
    best_path = os.path.join(LOG_DIR + f"_{method}", "checkpoints", "best_model.pth")
    model.load_state_dict(torch.load(best_path, map_location=device))

    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)
    test_loss, test_acc = validate_one_epoch(model, test_loader, criterion, device)

    print(f"{method} Val Acc: {val_acc:.4f} | Test Acc: {test_acc:.4f}")

    summary_rows.append({
        "method": method,
        "original_dim": X_train_std.shape[1],
        "reduced_dim": Xtr.shape[1],
        "val_acc": val_acc,
        "test_acc": test_acc
    })

    if test_acc > best_acc:
        best_acc = test_acc
        best_method = method
        best_data = (Xtr, Xva, Xte)


================ PCA ================

Starting the first epoch...
 Epoch 1/100 | Train Loss: 2.8735 | Train Acc: 0.7324 | Val Loss: 5.4293 | Val Acc: 0.1578
 Epoch 2/100 | Train Loss: 1.3919 | Train Acc: 0.9741 | Val Loss: 5.3297 | Val Acc: 0.1899
 Epoch 3/100 | Train Loss: 1.2125 | Train Acc: 0.9991 | Val Loss: 5.4929 | Val Acc: 0.1896
 Epoch 4/100 | Train Loss: 1.1816 | Train Acc: 0.9998 | Val Loss: 5.5777 | Val Acc: 0.1959
 Epoch 5/100 | Train Loss: 1.1673 | Train Acc: 0.9999 | Val Loss: 5.6067 | Val Acc: 0.1975
 Epoch 6/100 | Train Loss: 1.1575 | Train Acc: 1.0000 | Val Loss: 5.6425 | Val Acc: 0.2018
 Epoch 7/100 | Train Loss: 1.1496 | Train Acc: 1.0000 | Val Loss: 5.6657 | Val Acc: 0.2012
 Epoch 8/100 | Train Loss: 1.1452 | Train Acc: 1.0000 | Val Loss: 5.6787 | Val Acc: 0.1994
 Epoch 9/100 | Train Loss: 1.1412 | Train Acc: 1.0000 | Val Loss: 5.6946 | Val Acc: 0.2033
 Epoch 10/100 | Train Loss: 1.1381 | Train Acc: 1.0000 | Val Loss: 5.6966 | Val Acc: 0.2047
 Epoch 11/100 | Train

In [10]:
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(LOG_DIR+"fishvista_dim_reduction_summary.csv", index=False)

print("\nSaved summary to dim_reduction_summary.csv")
print(summary_df)


Saved summary to dim_reduction_summary.csv
         method  original_dim  reduced_dim   val_acc  test_acc
0           PCA          2375         1640  0.189851  0.212248
1           ICA          2375         1640  0.116831  0.150394
2  TruncatedSVD          2375         1640  0.184393  0.213884
3           GRP          2375         1640  0.180115  0.210101
4           SRP          2375         1640  0.182475  0.218178
5           NMF          2375         1640  0.155923  0.144975


In [11]:
print(f"\n🏆 Best method: {best_method} (Test Acc={best_acc:.4f})")

Xtr_best, Xva_best, Xte_best = best_data

cols = [f"{best_method}_comp_{i}" for i in range(Xtr_best.shape[1])]

train_out = pd.DataFrame(Xtr_best, columns=cols)
train_out["label"] = y_train

val_out = pd.DataFrame(Xva_best, columns=cols)
val_out["label"] = y_val

test_out = pd.DataFrame(Xte_best, columns=cols)
test_out["label"] = y_test

train_out.to_csv(LOG_DIR+f"fishvista_reduced_train_{best_method}.csv", index=False)
val_out.to_csv(LOG_DIR+f"fishvista_reduced_val_{best_method}.csv", index=False)
test_out.to_csv(LOG_DIR+f"fishvista_reduced_test_{best_method}.csv", index=False)

print("✅ Saved reduced datasets for best method.")


🏆 Best method: SRP (Test Acc=0.2182)
✅ Saved reduced datasets for best method.
